# Evaluation

Evaluating solutions of ill-defined problems using FP2MP-Eval approach

In [2]:
from src import read_json, write_json

solutions = read_json('./data/solutions.json')

In [4]:
from tqdm import tqdm
from fp2mp_eval.core import FP2MPEval

fp2mp_eval = FP2MPEval(n_judges=5)

for solution in tqdm(solutions):
    if 'Amsterdam. Develop a city branding strategy balancing tourism and local identity.' not in solution['problem']:
        break
    solution['evaluations'] = fp2mp_eval.evaluate_case((solution['problem'], solution['solution']), 5)

100%|██████████| 3/3 [03:12<00:00, 64.28s/it]


In [5]:
for solution in solutions:
    token_input = 0
    token_output = 0
    for m in solution['log']:
        response_metadata = m['response_metadata']
        if 'token_usage' in response_metadata:
            token_usage = response_metadata['token_usage']
            token_input += token_usage.get('prompt_tokens', 0)
            token_output += token_usage.get('completion_tokens', 0)
    solution['token_input'] = token_input
    solution['token_output'] = token_output
    solution['token_total'] = token_input + token_output

In [15]:
for solution in solutions:
    evaluations = solution['evaluations']
    evaluations_df = fp2mp_eval.evaluations_to_df(evaluations)
    evaluations_dict = evaluations_df.mean().to_dict()
    solution.update(evaluations_dict)